# Train the verdict model

Fine-tunes AfroXLMR to read a **claim** plus one piece of **evidence** and answer `supported`, `contradicted` or `not_enough_evidence`.

**Before you run it**
1. Runtime > Change runtime type > **T4 GPU**.
2. In Google Drive create a folder called `ethio-fakenews` and upload these into it:
   - `examples.jsonl` (the rule-made examples)
   - `gemini_examples.jsonl` (the examples Gemini wrote)
   - `gemini_audit.csv` (the test file)
3. Run the cells from top to bottom. The best model is saved to `ethio-fakenews/verdict-model` in your Drive after every epoch.

**What it trains on:** the Gemini examples if there are enough of them, otherwise the rule-made examples. **What it is tested on:** `gemini_audit.csv`.

**Training time = number of examples x number of epochs.** On a free T4 the large model processes roughly 15 to 30 examples a second (the log prints a better estimate after the first 100 steps).

In [ ]:
!pip -q install transformers sentencepiece protobuf scikit-learn pandas

## 1. Setup

In [ ]:
import json, math, os, random, time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ethio-fakenews"
OUT_DIR = f"{DRIVE_DIR}/verdict-model"

# What to train on: "auto" uses the Gemini examples when there are at least
# GEMINI_MIN_EXAMPLES of them, and the rule-made examples otherwise.
# You can also force "gemini" or "rules".
TRAIN_SOURCE = "auto"
GEMINI_MIN_EXAMPLES = 5000

# AfroXLMR large, pretrained on 61 languages including Amharic, Tigrinya and Oromo.
# If the GPU runs out of memory, lower BATCH_SIZE to 4 and raise ACCUM_STEPS to 4.
MODEL_NAME = "Davlan/afro-xlmr-large-61L"
MAX_LEN = 256
BATCH_SIZE = 8
ACCUM_STEPS = 2          # effective batch size = BATCH_SIZE * ACCUM_STEPS
EPOCHS = 4
LEARNING_RATE = 1e-5
WARMUP_FRACTION = 0.1
SEED = 42

# Training time = examples x epochs. Cap the training set so the run fits one Colab session.
MAX_TRAIN_EXAMPLES = 20000
MAX_VAL_EXAMPLES = 3000

LABELS = ["supported", "contradicted", "not_enough_evidence"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
if DEVICE == "cpu":
    print("WARNING: no GPU found. Training will be extremely slow. Change the runtime type to T4 GPU.")
print("Training on:", DEVICE)

## 2. Load the data

In [ ]:
def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def norm(text):
    return " ".join(str(text).split())


rule_rows = read_jsonl(f"{DRIVE_DIR}/examples.jsonl")
gemini_rows = read_jsonl(f"{DRIVE_DIR}/gemini_examples.jsonl")

# The test: gemini_audit.csv. Its label is the gemini_label column.
# If you filled in your_check, rows marked "wrong" are left out of the test.
audit = pd.read_csv(f"{DRIVE_DIR}/gemini_audit.csv", encoding="utf-8-sig")
if "your_check" in audit.columns:
    audit = audit[audit["your_check"].astype(str).str.strip().str.lower() != "wrong"]
audit["label"] = audit["gemini_label"].astype(str).str.strip().str.lower()
test_rows = audit[audit["label"].isin(LABELS)][["language", "claim", "evidence", "label"]].to_dict("records")

# Never train on anything built from the same articles as the test passages.
test_evidence = {norm(r["evidence"]) for r in test_rows}
test_articles = {r["article"] for r in rule_rows + gemini_rows if norm(r["evidence"]) in test_evidence}
rule_rows = [r for r in rule_rows if r["article"] not in test_articles]
gemini_rows = [r for r in gemini_rows if r["article"] not in test_articles]

use_gemini = TRAIN_SOURCE == "gemini" or (TRAIN_SOURCE == "auto" and len(gemini_rows) >= GEMINI_MIN_EXAMPLES)
pool = gemini_rows if use_gemini else rule_rows
print(f"Gemini examples available: {len(gemini_rows)} | rule-made examples available: {len(rule_rows)}")
print("Training on:", "the Gemini examples" if use_gemini else "the rule-made examples")
if not pool:
    raise SystemExit("There are no examples to train on. Check the files in your Drive folder.")

# About one article in ten is kept aside to check progress after every epoch.
val_rows = [r for r in pool if int(r["article"], 16) % 10 == 1]
train_rows = [r for r in pool if int(r["article"], 16) % 10 != 1]
if len(train_rows) > MAX_TRAIN_EXAMPLES:
    train_rows = random.sample(train_rows, MAX_TRAIN_EXAMPLES)
if len(val_rows) > MAX_VAL_EXAMPLES:
    val_rows = random.sample(val_rows, MAX_VAL_EXAMPLES)

print(f"using: train {len(train_rows)} | validation {len(val_rows)} | test {len(test_rows)}")
print("train labels:   ", dict(Counter(r["label"] for r in train_rows)))
print("train languages:", dict(Counter(r["language"] for r in train_rows)))
print("test languages: ", dict(Counter(r["language"] for r in test_rows)))

## 3. Turn text into model input

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)


class PairDataset(Dataset):
    """Each item is (claim, evidence) joined into one input, plus its label."""

    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        item = self.items[i]
        encoded = tokenizer(item["claim"], item["evidence"], truncation="only_second", max_length=MAX_LEN)
        return {**encoded, "labels": LABEL2ID[item["label"]]}


def make_loader(items, shuffle, batch_size=BATCH_SIZE):
    return DataLoader(PairDataset(items), batch_size=batch_size, shuffle=shuffle, collate_fn=collator)


train_loader = make_loader(train_rows, shuffle=True)

# Check how long the inputs are, so MAX_LEN is not cutting much off.
sample = random.sample(train_rows, min(300, len(train_rows)))
lengths = [len(tokenizer(r["claim"], r["evidence"])["input_ids"]) for r in sample]
print("input length in tokens: median", int(np.median(lengths)), "| 95th percentile", int(np.percentile(lengths, 95)), "| MAX_LEN", MAX_LEN)

## 4. Helpers to measure the model

In [ ]:
@torch.no_grad()
def predict(model, items):
    model.eval()
    predictions = []
    for batch in make_loader(items, shuffle=False, batch_size=32):
        batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        predictions.extend(logits.argmax(dim=-1).cpu().tolist())
    return np.array(predictions)


def report(items, predictions, title):
    truth = np.array([LABEL2ID[r["label"]] for r in items])
    print(f"\n=== {title} ({len(items)} examples) ===")
    print(f"accuracy {accuracy_score(truth, predictions):.3f} | macro-F1 {f1_score(truth, predictions, average='macro', labels=[0, 1, 2], zero_division=0):.3f}")
    languages = np.array([r["language"] for r in items])
    for language in sorted(set(languages)):
        mask = languages == language
        f1 = f1_score(truth[mask], predictions[mask], average="macro", labels=[0, 1, 2], zero_division=0)
        print(f"  {language}: {mask.sum():4d} examples | accuracy {accuracy_score(truth[mask], predictions[mask]):.3f} | macro-F1 {f1:.3f}")
    print("confusion matrix (rows = true, columns = predicted, order:", ", ".join(LABELS), ")")
    print(confusion_matrix(truth, predictions, labels=[0, 1, 2]))
    return f1_score(truth, predictions, average="macro", labels=[0, 1, 2], zero_division=0)

## 5. Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS)).to(DEVICE)

# Rarer labels get a bigger weight, so the model cannot win by ignoring them.
counts = Counter(r["label"] for r in train_rows)
weights = torch.tensor([len(train_rows) / (len(LABELS) * max(counts[label], 1)) for label in LABELS], dtype=torch.float).to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best_f1 = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    started = time.time()
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, start=1):
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), labels) / ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 100 == 0:
            seconds_per_step = (time.time() - started) / step
            minutes_left = seconds_per_step * (len(train_loader) - step) / 60
            print(f"epoch {epoch}/{EPOCHS} step {step}/{len(train_loader)} loss {running_loss / step:.4f} | about {minutes_left:.0f} min left in this epoch")

    val_f1 = report(val_rows, predict(model, val_rows), f"validation after epoch {epoch}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained(OUT_DIR)
        tokenizer.save_pretrained(OUT_DIR)
        print(f"saved the best model so far to {OUT_DIR}")

## 6. Final test
The test passages and the articles they came from were kept out of training. The labels in `gemini_audit.csv` come from Gemini.

In [ ]:
best = AutoModelForSequenceClassification.from_pretrained(OUT_DIR).to(DEVICE)
report(test_rows, predict(best, test_rows), "test on gemini_audit.csv")